In [1]:
import sqlite3
import pandas as pd
import os
from core.variables.variable import DATA_DIRECTORY,COLUMNS_STRUCTURE,REVENU,DEPENSE

def explorer_bdd(nom_db: str):
    """
    Affiche le contenu de toutes les tables d'une base de données SQLite.
    Input: nom de la base (ex: "MaCompta_Test")
    """
    # Construction du chemin vers le dossier _data
    path_db = os.path.join(DATA_DIRECTORY, f"{nom_db}.db")
    
    if not os.path.exists(path_db):
        print(f"Erreur : La base de données '{path_db}' est introuvable.")
        return

    try:
        # Connexion à la base
        conn = sqlite3.connect(path_db)
        
        # 1. Lister toutes les tables existantes
        query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
        tables = pd.read_sql(query_tables, conn)['name'].tolist()
        
        if not tables:
            print(f"La base de données '{nom_db}' est vide (aucune table).")
            return

        print(f"=== EXPLORATION DE LA BDD : {nom_db} ===")
        print(f"Tables trouvées : {', '.join(tables)}\n")

        # 2. Afficher le contenu de chaque table
        for table in tables:
            print(f"--- Table : {table} ---")
            df = pd.read_sql(f"SELECT * FROM {table}", conn)
            
            if df.empty:
                print("Table vide.")
            else:
                # On affiche les 10 premières lignes pour ne pas encombrer le notebook
                display(df) # Utilise display() qui est plus propre que print() dans un Notebook
            print("\n")

        conn.close()
        
    except Exception as e:
        print(f"Une erreur est survenue lors de la lecture : {e}")

# --- EXEMPLE D'UTILISATION DANS TON NOTEBOOK ---
explorer_bdd("Compte")

=== EXPLORATION DE LA BDD : Compte ===
Tables trouvées : CH_compte_courant, Compte_Courant, Compte_Jeune, Livret_Bleu, _metadata_accounts

--- Table : CH_compte_courant ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,2026-04-16,Assurance swisscare,Essentiel,Assurance,Depense,38.00,ac15142b
1,2026-04-16,SBB CHF,Voyage,SBB,Depense,62.00,f6711d4c
2,2026-04-14,course,Essentiel,Nourriture,Depense,62.68,18bd1a37
3,2026-04-12,Erreur lié a l'app v4,Erreur,Erreur,Depense,267.00,4ed0675b
4,2026-04-07,virement,Virement,Compte_Courant,Depense,2000.00,44f5990b
5,2026-04-03,Plein d essence,Voyage,voiture,Depense,59.38,fda2253d
6,2026-04-03,burger King,Loisir,Nourriture,Depense,22.72,1d34e3e6
7,2026-04-01,Loyer,Essentiel,Logement,Depense,1400.00,417811f2
8,2026-03-31,course,Essentiel,Nourriture,Depense,179.97,1ad9e8f2
9,2026-03-28,Framboisier + Blue berry,Loisir,Plante,Depense,51.10,705fb0d1




--- Table : Compte_Courant ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,2026-04-17,Erreur lié à la mise en place de la version V4...,Erreur de compte,Erreur de compte,Depense,518.68,723ef1ad
1,2026-04-14,PinkMin mania,Loisir,Jeux Vidéo,Depense,5.99,1e0edc98
2,2026-04-10,Photo remise des diplomes,Special,Cadeau,Depense,12.89,7195e23d
3,2026-04-08,virement travail,Virement,Livret_Bleu,Depense,2500.00,6aa1b443
4,2026-04-08,"Achat de cable ,...",Essentiel,Fourniture,Depense,34.82,44c287c9
...,...,...,...,...,...,...,...
430,2025-02-10,Loyer,Habitation,Loyer,Depense,229.00,4bda8cb8
431,2025-02-10,Autoentreprenariat,Travail,Fiver,Revenu,6.99,4a527ab2
432,2025-02-10,Eau,Habitation,Charge,Depense,19.00,5aa7ff16
433,2025-02-10,Spotify,Loisir,Musique,Depense,6.99,e1b045e7




--- Table : Compte_Jeune ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,2026-07-01,Interet,Banque,Interet,Revenu,66.60,eafa261f
1,2025-02-09,initialisation,Banque,Intialisation,Revenu,1665.15,2c0d8cee




--- Table : Livret_Bleu ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,2026-04-17,Erreur lie a la v4,Erreur,Erreur,Depense,40.00,00dd5ddd
1,2026-04-08,Virement,Banque,Compte_Courant,Revenu,2500.00,3e5928e4
2,2026-04-03,Virement,Banque,Compte_Courant,Depense,150.00,507b096c
3,2026-03-01,Virement,Banque,Compte_Courant,Depense,800.00,3c7a1b31
4,2026-01-07,Virement,Banque,Compte_Courant,Depense,1300.00,7c666f79
5,2026-01-05,Virement,Banque,Compte_Courant,Depense,4000.00,0b966402
6,2026-01-01,Interet,Interet,Banque,Revenu,93.69,ed7fffdb
7,2026-01-01,Virement,Banque,Compte_Courant,Depense,3000.00,69583eb9
8,2025-12-28,Virement,Banque,Compte_Courant,Revenu,300.00,e111dee4
9,2025-12-20,Virement,Banque,Compte_Courant,Revenu,4000.00,26766e3d




--- Table : _metadata_accounts ---


,account_name,devise,state
0,CH_compte_courant,CHF,1
1,Compte_Courant,€,1
2,Compte_Jeune,€,1
3,Livret_Bleu,€,1


In [2]:
import os
from flask import Flask, render_template, send_from_directory
from core.persistence.database_manager import DatabaseManager
db_manager = DatabaseManager("Compte") 

In [3]:
compta= db_manager.comptabilite

In [4]:
for i in range(len(compta.liste_compte)):
    print(compta.liste_compte[i].account_name)
n = 1
print(compta.liste_compte[n].account_name)

df = compta.liste_compte[n].df

CH_compte_courant
Compte_Courant
Compte_Jeune
Livret_Bleu
Compte_Courant


In [5]:
import plotly.graph_objects as go
import pandas as pd

def generer_sankey_entonnoir_pur(df_raw,name):
    df = df_raw.copy()
    df['Valeur'] = df['Valeur'].abs()

    # 1. Extraction des deux types
    df_rev = df[df['Type'] == 'Revenu']
    df_dep = df[df['Type'] == 'Depense']

    # --- ÉTAPE 1 : REVENUS -> POINT CENTRAL ---
    flux_entree = df_rev.groupby('Categorie')['Valeur'].sum().reset_index()
    flux_entree.columns = ['source', 'value']
    flux_entree['target'] = 'PORTEFEUILLE'
    # Suffixe pour éviter les collisions de noms entre revenu et dépense
    flux_entree['source'] = flux_entree['source'] + " " 

    # --- ÉTAPE 2 : POINT CENTRAL -> DÉPENSES ---
    flux_sortie = df_dep.groupby('Categorie')['Valeur'].sum().reset_index()
    flux_sortie.columns = ['target', 'value']
    flux_sortie['source'] = 'PORTEFEUILLE'

    # Fusion des flux
    flux_total = pd.concat([flux_entree, flux_sortie], ignore_index=True)

    # 2. Indexation des noeuds
    nodes = list(pd.unique(flux_total[['source', 'target']].values.ravel('K')))
    mapping = {node: i for i, node in enumerate(nodes)}
    
    flux_total['source_idx'] = flux_total['source'].map(mapping)
    flux_total['target_idx'] = flux_total['target'].map(mapping)

    # 3. Couleurs pour un look "Premium Dashboard"
    node_colors = []
    for node in nodes:
        if node == 'PORTEFEUILLE':
            node_colors.append("#00adb5") # Ton cyan fétiche pour le centre
        elif node.endswith(" "):
            node_colors.append("#2ecc71") # Vert pour les Revenus
        else:
            node_colors.append("#e74c3c") # Rouge pour les Dépenses

    # 4. Création du graphique
    fig = go.Figure(data=[go.Sankey(
        arrangement = "snap",
        node = dict(
          pad = 25,
          thickness = 40,
          label = [n.strip() for n in nodes], # On enlève l'espace pour l'affichage
          color = node_colors,
          line = dict(color = "rgba(0,0,0,0)", width = 0)
        ),
        link = dict(
          source = flux_total['source_idx'],
          target = flux_total['target_idx'],
          value = flux_total['value'],
          color = "rgba(144, 148, 151, 0.25)" # Liens discrets
        )
    )])

    fig.update_layout(
        title_text=f"FLUX DE TRÉSORERIE {name}",
        template="plotly_dark",
        font_size=14,
        height=650,
        margin=dict(l=10, r=10, t=50, b=10)
    )
    
    fig.show()
nb = 0
# --- TEST ---
generer_sankey_entonnoir_pur(compta.liste_compte[nb].df,compta.liste_compte[nb].account_name  )